# 06 · Un device creato direttamente in ADR

Finora ogni device è nato da una registrazione a DPS. Qui facciamo il contrario: **scriviamo il device nel namespace ADR, da soli**, senza passare da un IoT Hub e senza Azure IoT Operations.

La domanda: cosa ci dà ADR da solo, e cosa no?
- **Sì**: un'anagrafica. Nome, produttore, modello, software revision, tag, stato abilitato o disabilitato.
- **No**: le credenziali. Lo dice l'aiuto della CLI: *"la creazione del device non emette credenziali"*. I profili di autenticazione li "materializzano" i servizi di provisioning.

**Serve:** i notebook 01 e 02a-02c già eseguiti (serve il namespace). Il notebook crea due device `autonomo-01` e `autonomo-02`, un gruppo di prova, e alla fine **cancella tutto quello che ha creato**. Non tocca gli altri device.

**Da dire.** "ADR è prima di tutto un catalogo, e il catalogo si scrive da solo. La fiducia, cioè i certificati, è un'altra cosa: arriva dal provisioning."

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

In [ ]:
$nomi = 'autonomo-01', 'autonomo-02'     # i device che creiamo (e cancelliamo alla fine)
$gruppo = 'wpc-g-autonomi'                # il gruppo di prova

## 1 · Creare i device
Un solo comando per device: `az iot adr ns device create`. È "crea o sostituisci", quindi la cella si può rieseguire.
`--external-device-id` è l'identificativo che il cliente dà al device (per esempio quello di fabbrica): **si scrive solo alla creazione**.

In [ ]:
foreach ($n in $nomi) {
    az iot adr ns device create `
        --name $n `
        --namespace $adrName `
        --resource-group $rg `
        --external-device-id "fabbrica-$n" `
        --manufacturer Contoso `
        --model SensorV1 `
        --software-revision 1.0.0 `
        --hardware-revision A `
        --tags fleet=autonomi `
        -o none `
        --only-show-errors
    Assert-Az "Creazione di $n"
    "$n creato."
}

## 2 · Cosa c'è nel namespace
I due device ci sono, con le loro proprietà, e **sono risorse Azure** come le altre (portale, `az resource`, Azure Policy).

In [ ]:
az iot adr ns device list `
    --namespace $adrName `
    --resource-group $rg `
    --query "[?starts_with(name,'autonomo-')].{device:name, produttore:properties.manufacturer, modello:properties.model, softwareRevision:properties.softwareRevision, stato:properties.enablementState}" `
    -o table `
    --only-show-errors
Assert-Az 'Elenco dei device'

Un device per intero, come lo restituisce il servizio:

In [ ]:
az iot adr ns device show `
    --name $nomi[0] `
    --namespace $adrName `
    --resource-group $rg `
    -o jsonc `
    --only-show-errors
Assert-Az "Lettura di $($nomi[0])"

## 3 · Non hanno credenziali
Il device esiste, ma **nessuno può usarlo per collegarsi**: non ha un certificato, non ha un Hub. Se i profili di autenticazione sono vuoti, è il punto della slide: ADR dà l'identità, non la fiducia.

In [ ]:
az iot adr ns device auth list `
    --device-name $nomi[0] `
    --namespace $adrName `
    --resource-group $rg `
    -o jsonc `
    --only-show-errors
Assert-Az 'Elenco dei profili di autenticazione'

## 4 · Cambiare il device
Le proprietà si cambiano con `update`, e lo stato abilitato o disabilitato è una proprietà come le altre. Qui: nuova software revision per il primo, il secondo disabilitato.

In [ ]:
Set-AdrDeviceProperties -Name $nomi[0] -Namespace $adrName -ResourceGroup $rg -SoftwareRevision '1.1.0'
"stato di $($nomi[1]) dopo la disabilitazione: " + (Set-AdrDeviceEnablement -Name $nomi[1] -Namespace $adrName -ResourceGroup $rg -State Disabled)

In [ ]:
az iot adr ns device list `
    --namespace $adrName `
    --resource-group $rg `
    --query "[?starts_with(name,'autonomo-')].{device:name, softwareRevision:properties.softwareRevision, stato:properties.enablementState}" `
    -o table `
    --only-show-errors
Assert-Az 'Elenco dei device'

## 5 · Un gruppo li vede?
Proviamo. **Non so cosa risponde il servizio**: non l'ho mai provato su device creati così. La documentazione dice che i job coprono solo device collegati a un IoT Hub, quindi per i job probabilmente no; per i gruppi, che sono solo una query sulle proprietà, forse sì.
La cella stampa la risposta, qualunque sia. Il gruppo è una fotografia: se i membri sono zero, può essere questione di tempo.

In [ ]:
$errore = New-AdrGroup -Name $gruppo -Query "WHERE model = 'SensorV1'" -Namespace $adrName -ResourceGroup $rg
if ($errore) { "Il servizio ha rifiutato il gruppo: $errore" }
else {
    "Gruppo creato."
    "Membri: " + ((Get-AdrGroupMembers -Name $gruppo -Namespace $adrName -ResourceGroup $rg) -join ', ')
}

## Pulizia
Cancella il gruppo e i due device. Non tocca nient'altro nel namespace.

In [ ]:
Remove-AdrGroup -Name $gruppo -Namespace $adrName -ResourceGroup $rg
foreach ($n in $nomi) {
    az iot adr ns device delete `
        --name $n `
        --namespace $adrName `
        --resource-group $rg `
        --yes `
        --only-show-errors
    Assert-Az "Eliminazione di $n"
}
"Eliminati: " + ($nomi -join ', ') + " e il gruppo $gruppo."

## Cosa abbiamo visto
- Un device si può **creare direttamente in ADR**, senza Hub e senza Azure IoT Operations: è una risorsa del namespace, con proprietà, tag e stato.
- Creato così **non ha credenziali**: certificati e profili di autenticazione arrivano dai servizi di provisioning (DPS).
- ADR è l'**anagrafica**; la **fiducia** è un'altra cosa. È la stessa distinzione della slide "Cosa risolve ADR, cosa resta a te".

**Non provato** (da scoprire con questo notebook o a mano):
- se un gruppo vede i device creati così;
- cosa succede se uno di questi device si registra poi a DPS con lo stesso nome (se DPS lo adotta o dà errore);
- se un job può raggiungerli (la documentazione dice di no, solo device collegati a un Hub).